# 评估模型

## Geo-Model

In [2]:
import os
import torch
import numpy as np
from tqdm import tqdm
from DATASET_PATH import DATASET_PATH
from dataloader import get_test_loader
from model_unet import Gravity_Inverse_UNet_2D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_root_path = "checkpoints/geo_model"
save_root_path = "test_result/geo_model/"
model_name = "last_model"

# 构建测试数据加载器
test_loaders = []
for name, zarr_file_path in DATASET_PATH['geo_model'].items():
    test_loaders.append(get_test_loader(
        zarr_file_path, 
        batch_size=32, 
        num_workers=8
    ))

# 加载模型
model = Gravity_Inverse_UNet_2D()
model.load_state_dict(torch.load(os.path.join(model_root_path, f"{model_name}.pth"))["model_state_dict"])

# 评估模型
model.eval()
model.to(device)
for idx, test_loader in enumerate(test_loaders):
    with torch.no_grad():
        data_list = []
        label_list = []
        pred_list = []
        bar = tqdm(test_loader, desc=f"Test for test_loader {idx}")
        for _, (data, label) in enumerate(bar):
            data = data.to(device)
            pred = model(data)

            data_list.extend(data.to("cpu").numpy())
            label_list.extend(label.to("cpu").numpy())
            pred_list.extend(pred.to("cpu").numpy())
            bar.update(1)

        bar.close()

        save_path = os.path.join(save_root_path, f"{model_name}", f"test_result_{idx+1}.npz")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        np.savez(
            save_path, 
            data=np.array(data_list), 
            label=np.array(label_list), 
            pred=np.array(pred_list)
        )

Test for test_loader 4: 100%|██████████| 50/50 [00:04<00:00, 11.46it/s]


# 绘制对比图

In [1]:
import os
import numpy as np
from matplotlib import pyplot as plt

def load_test_result(result_path):
    result = np.load(result_path)
    data, label, pred = result['data'], result['label'], result['pred']
    nz, nx = label.shape[-2:]
    label = label.reshape(-1, nz, nx)
    pred = pred.reshape(-1, nz, nx)
    return data, label, pred

def plot_test_result(label, pred, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    density_img = axes[0].imshow(label, cmap="viridis", aspect='auto', vmin=0, vmax=1)
    axes[0].set_title("True Density Model", fontsize=12)
    axes[0].set_xlabel("X (Grid Points)")
    axes[0].set_ylabel("Z (Grid Points)")
    plt.colorbar(density_img, ax=axes[0])

    density_img = axes[1].imshow(pred, cmap="viridis", aspect='auto', vmin=0, vmax=1)
    axes[1].set_title("Pred Density Model", fontsize=12)
    axes[1].set_xlabel("X (Grid Points)")
    axes[1].set_ylabel("Z (Grid Points)")
    plt.colorbar(density_img, ax=axes[1])

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path)
    else:
        plt.show()
    
    plt.close()
    plt.clf()


In [4]:
model_root_path = "test_result/geo_model/"
model_type = "last_model"
test_result_list = [
    os.path.join(model_root_path, model_type, f"test_result_{i}.npz") 
    for i in range(1, 6)
]

for test_result in test_result_list:
    data, label, pred = load_test_result(test_result)
    for idx in range(100):
        _label = label[idx]
        _pred = pred[idx]
        plot_test_result(
            _label, 
            _pred, 
            save_path=os.path.join(
                model_root_path, 
                model_type,
                os.path.basename(test_result).split(".")[0],
                f"{idx:03d}.png")
            )

<Figure size 640x480 with 0 Axes>